# WARN Layoff Notices API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/warn-layoff-alerts.ipynb)

**What:** WARN Act layoff notices from 7 official state sources (TX, CA, WA, OR, MD, VA, NC), verified by 89 passing tests. Clean rows: employer, workers affected, city, notice and layoff dates. Deduped across runs for scheduled alerts. CSV, JSON, API or webhook. $0.003 per notice.

**Why:** run the [`warn-layoff-alerts`](https://apify.com/transparent_meteorite/warn-layoff-alerts) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Company-level notices only; no contact data is displayed.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/warn-layoff-alerts) for every option.

In [2]:
run_input = {'states': ['TX'], 'daysBack': 365, 'maxResults': 5, 'onlyNewSinceLastRun': False}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/warn-layoff-alerts').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> Status: RUNNING, Message: 


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:30.319Z ACTOR: Pulling container image of build iUtAFu7EgS5SnlrTt from registry.


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:30.322Z ACTOR: Creating container.


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:30.393Z ACTOR: Starting container.


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:30.394Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:31.816Z [warn] Done. Scanned 112, delivered 5 new notices (enriched 0).


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:31.892Z INFO  [Status message]: Done: 5 items


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> 2026-10-13T22:19:31.931Z Thanks for using WARN Layoff Notices. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/sEhV6snEeCTOKAJfx/issues) with the run URL and we will reply there.


[apify.warn-layoff-alerts runId:GTuPB9Vi0SKgpsEJn] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

Company-level notices only; no contact data is displayed.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["company", "city", "county", "state", "layoff_count", "notice_date", "layoff_date", "wda"] if c in df.columns]
df[SHOW]

,company,city,county,state,layoff_count,notice_date,layoff_date,wda
0,JPMorgan Chase & Co.,Plano,Collin,TX,244,2026-06-23,2026-08-21,North Central Texas WDA
1,KUEHNE + NAGEL (KN) 2026,Lewisville,Denton,TX,90,2026-06-16,2026-06-29,North Central Texas WDA
2,"Fortrex (John Soules Foods, Inc.)",Tyler,Smith,TX,85,2026-06-12,2026-06-26,East Texas WDA
3,First Student (DeSoto I.S.D.),De Soto,Dallas,TX,74,2026-06-12,2026-06-30,Dallas County WDA
4,SAKS GLOBAL (Neiman Marcus),Dallas,Dallas,TX,67,2026-06-12,2026-09-30,Dallas County WDA


## 5. A quick analysis

In [5]:
df['layoff_count']=pd.to_numeric(df['layoff_count'],errors='coerce')
print('Total workers in sample:',int(df['layoff_count'].sum()))
df.groupby('city')['layoff_count'].sum().sort_values(ascending=False)

Total workers in sample: 560


city
Plano         244
Lewisville     90
Tyler          85
De Soto        74
Dallas         67
Name: layoff_count, dtype: int64

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [warn-layoff-alerts on the Apify Store](https://apify.com/transparent_meteorite/warn-layoff-alerts).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.